# Gemini 2.5 Flash — Adapted (300) vs Original Comparison

**Goal:** Test whether Gemini's high accuracy on adapted puzzles is genuine or a subset composition effect.

**Pipeline:**
1. Sample 300 adapted puzzles from the judge-confirmed pool
2. Run CoT (thinking enabled) on adapted puzzles only
3. Semantic parser audit to extract answers consistently
4. Load existing original results from `all_models_audited.csv`
5. Pair on `(Puzzle_Number, Question_Number)` and compare
6. Paired t-test + Wilcoxon + difficulty breakdown

**Note:** Original results are NOT re-run — we use the existing `all_models_audited.csv` to avoid changing any prior results.

## 1. Install & Imports

In [ ]:
!pip install -q google-genai pandas openpyxl tqdm matplotlib scipy

In [ ]:
import re, os, time, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 9, 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'axes.linewidth': 0.8, 'figure.dpi': 150,
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import pearsonr, ttest_rel, wilcoxon
from tqdm import tqdm
from google import genai
from google.genai import types

## 2. Config

In [ ]:
# ── API ───────────────────────────────────────────────────────────────────────
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', 'YOUR_KEY_HERE')
MODEL_ID       = 'gemini-2.5-flash'

# ── Paths ─────────────────────────────────────────────────────────────────────
# Original puzzle dataset — for GT_Answer + Solved_Percentile
GT_PATH          = '/content/puzzle_samples.xlsx'

# Adapted puzzles — results_combined_audited with final_adapted_puzzle column
ADAPTED_PATH     = '/content/results_combined_audited.csv'

# Existing Gemini original results — do NOT re-run these
ORIG_AUDITED_PATH = '/content/all_models_audited_original.csv'
ORIG_MODEL_NAME   = 'gemini-2.5-flash'   # must match model_name in that file

WORKDIR = './outputs_gemini_300'
os.makedirs(WORKDIR, exist_ok=True)

# ── Sampling ──────────────────────────────────────────────────────────────────
N_SAMPLE    = 300
RANDOM_SEED = 42

# ── Generation ────────────────────────────────────────────────────────────────
SAVE_EVERY          = 50
MAX_OUTPUT_COT      = 5000
THINKING_BUDGET_COT = -1   # dynamic thinking

# ── Semantic parser audit ─────────────────────────────────────────────────────
MAX_OUTPUT_AUDIT  = 256
THINKING_BUDGET_AUDIT = 0  # no thinking for answer extraction
ONLY_MISSING      = True   # only audit rows where answer wasn't parsed

# ── Output files ──────────────────────────────────────────────────────────────
COT_ADAP_OUT   = os.path.join(WORKDIR, 'cot_adapted_gemini25flash.csv')
AUDIT_OUT      = os.path.join(WORKDIR, 'cot_adapted_audited.csv')

KEY_COLS      = ['Puzzle_Number', 'Question_Number']
VALID_ANSWERS = {'A', 'B', 'C', 'D', 'E'}
PASS_OUTCOMES = ['equivalent', 'rescued_formatting']

## 3. Init Gemini Client

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)
print(f'Client ready. Model: {MODEL_ID}')

## 4. Load Data & Sample 300 Adapted Puzzles

In [ ]:
# ── Ground truth ──────────────────────────────────────────────────────────────
if GT_PATH.endswith('.xlsx'):
    gt_raw = pd.read_excel(GT_PATH)
else:
    gt_raw = pd.read_csv(GT_PATH)

gt = gt_raw[['Puzzle_Number', 'Question_Number', 'Answer', 'Solved_Percentile']].drop_duplicates(
    subset=['Puzzle_Number', 'Question_Number']
).rename(columns={'Answer': 'GT_Answer'}).copy()

def map_difficulty(p):
    if   p < 30: return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

gt['difficulty'] = gt['Solved_Percentile'].apply(map_difficulty)
print(f'Ground truth: {len(gt)} puzzles')
print(gt['difficulty'].value_counts().to_string())

# ── Adapted puzzles ───────────────────────────────────────────────────────────
adapted_raw = pd.read_csv(ADAPTED_PATH)
adapted_raw = adapted_raw[adapted_raw['corrected_outcome'].isin(PASS_OUTCOMES)].copy()
adapted_raw = adapted_raw.rename(columns={'final_adapted_puzzle': 'Sample'})

# Merge in GT_Answer + difficulty
adapted = adapted_raw.merge(
    gt[['Puzzle_Number', 'Question_Number', 'GT_Answer', 'Solved_Percentile', 'difficulty']],
    on=['Puzzle_Number', 'Question_Number'], how='left'
)[['Puzzle_Number', 'Question_Number', 'Sample', 'GT_Answer',
   'Solved_Percentile', 'difficulty', 'generator_model']].dropna(subset=['GT_Answer']).reset_index(drop=True)

print(f'\nAdapted pool (judge-confirmed): {len(adapted)} rows')
print(adapted['difficulty'].value_counts().to_string())
print('\nGenerator breakdown:')
print(adapted['generator_model'].value_counts().to_string())

# ── Stratified sample ─────────────────────────────────────────────────────────
tiers  = adapted['difficulty'].value_counts(normalize=True)
frames = []
for tier, prop in tiers.items():
    n_tier = round(prop * N_SAMPLE)
    pool   = adapted[adapted['difficulty'] == tier]
    n_tier = min(n_tier, len(pool))
    frames.append(pool.sample(n=n_tier, random_state=RANDOM_SEED))

sample = pd.concat(frames).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
if len(sample) < N_SAMPLE:
    extra = adapted[~adapted.index.isin(sample.index)]
    sample = pd.concat([sample, extra.sample(n=N_SAMPLE - len(sample), random_state=RANDOM_SEED)])
sample = sample.head(N_SAMPLE).reset_index(drop=True)

sample.to_csv(os.path.join(WORKDIR, 'sample_adapted_300.csv'), index=False)
print(f'\nSample: {len(sample)} adapted puzzles')
print(sample['difficulty'].value_counts().to_string())

## 5. Generation Helpers

In [ ]:
COT_SYSTEM_PROMPT = """\
You are an expert puzzle solver. Your task is to answer logical multiple-choice puzzles.
Each puzzle consists of three parts: <context>, <clues>, and <question>.

Instructions:
- Read the puzzle carefully.
- Think through the problem carefully before answering.
- Output only the correct option letter (A, B, C, D, or E). Nothing else.
"""

def extract_answer(text: str):
    if not isinstance(text, str) or not text.strip():
        return None
    for pat in [r'Answer:\s*([A-E])\b', r'\*\*([A-E])\*\*', r'\b([A-E])\b']:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return m.group(1).upper()
    return None

def save_results(records, path):
    pd.DataFrame(records).to_csv(path, index=False)

def load_completed(path, key_cols):
    if not os.path.exists(path):
        return [], set()
    done_df   = pd.read_csv(path)
    done_keys = set(zip(*[done_df[c] for c in key_cols]))
    records   = done_df.to_dict('records')
    print(f'  Resuming: {len(records)} rows already done.')
    return records, done_keys

def generate_cot(user_prompt):
    max_retries, wait = 5, 30
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=COT_SYSTEM_PROMPT,
                    max_output_tokens=MAX_OUTPUT_COT,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=THINKING_BUDGET_COT,
                        include_thoughts=True,
                    ),
                ),
            )
            elapsed = (time.time() - t0) / 60.0
            raw, reasoning = '', None
            for part in response.candidates[0].content.parts:
                if not part.text:
                    continue
                if part.thought:
                    reasoning = part.text.strip()
                else:
                    raw += part.text
            return raw.strip(), reasoning, elapsed
        except Exception as e:
            if attempt < max_retries - 1 and ('503' in str(e) or '429' in str(e)):
                print(f'  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s')
                time.sleep(wait)
                wait *= 2
            else:
                raise

## 6. Run CoT on Adapted Puzzles

In [ ]:
# Key includes Sample to handle multiple adaptations of same puzzle
RUN_KEY_COLS = ['Puzzle_Number', 'Question_Number', 'Sample']
cot_records, cot_done = load_completed(COT_ADAP_OUT, RUN_KEY_COLS)

for _, row in tqdm(sample.iterrows(), total=len(sample), desc='CoT Adapted'):
    key = (row['Puzzle_Number'], row['Question_Number'], row['Sample'])
    if key in cot_done:
        continue

    raw, reasoning, elapsed, answer = '', None, 0.0, None
    try:
        raw, reasoning, elapsed = generate_cot(row['Sample'])
        answer = extract_answer(raw)
    except Exception as e:
        raw = f'ERROR: {e}'
        print(f'  [P{row["Puzzle_Number"]} Q{row["Question_Number"]}] Error: {e}')

    cot_records.append({
        'Puzzle_Number':   row['Puzzle_Number'],
        'Question_Number': row['Question_Number'],
        'Sample':          row['Sample'],
        'GT_Answer':       row['GT_Answer'],
        'difficulty':      row['difficulty'],
        'Solved_Percentile': row['Solved_Percentile'],
        'generator_model': row['generator_model'],
        'Raw_Output':      raw,
        'Reasoning':       reasoning,
        'Model_Answer':    answer,
        'Inference_Min':   round(elapsed, 3),
    })
    cot_done.add(key)

    if len(cot_records) % SAVE_EVERY == 0:
        save_results(cot_records, COT_ADAP_OUT)
        print(f'  [Checkpoint] {len(cot_records)} rows saved.')

save_results(cot_records, COT_ADAP_OUT)
cot_df = pd.DataFrame(cot_records)
parsed = cot_df['Model_Answer'].notna().sum()
print(f'CoT Adapted done. {parsed}/{len(cot_df)} parsed by regex.')

## 7. Semantic Parser Audit — Extract Missing Answers

In [ ]:
AUDIT_SYSTEM_PROMPT = """\
You are extracting the final answer from a model's response to a multiple-choice logic puzzle.
The answer must be one of: A, B, C, D, or E.

Instructions:
- Look through the entire text provided (raw output and reasoning if present).
- Find the model's final committed answer choice.
- If the model changed its mind, use the LAST answer it committed to.
- If the model clearly reached an answer, output just that single letter.
- If the model did not reach any answer or only expressed uncertainty without committing, output: NONE

Output ONLY a single letter (A/B/C/D/E) or the word NONE. Nothing else.
"""

def generate_audit(user_prompt):
    max_retries, wait = 5, 30
    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=AUDIT_SYSTEM_PROMPT,
                    max_output_tokens=MAX_OUTPUT_AUDIT,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(thinking_budget=THINKING_BUDGET_AUDIT),
                ),
            )
            elapsed = (time.time() - t0) / 60.0
            raw = ''
            for part in response.candidates[0].content.parts:
                if not getattr(part, 'thought', False):
                    raw += (part.text or '')
            return raw.strip(), elapsed
        except Exception as e:
            if attempt < max_retries - 1 and ('503' in str(e) or '429' in str(e)):
                print(f'  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s')
                time.sleep(wait)
                wait *= 2
            else:
                raise

def parse_answer(raw: str) -> str:
    raw = raw.strip().upper()
    if raw in VALID_ANSWERS:
        return raw
    m = re.search(r'\b([ABCDE])\b', raw)
    if m:
        return m.group(1)
    return 'UNKNOWN'

# ── Run audit ─────────────────────────────────────────────────────────────────
cot_df = pd.read_csv(COT_ADAP_OUT)
audit_records, audit_done = load_completed(AUDIT_OUT, RUN_KEY_COLS)

print(f'Rows to process: {len(cot_df)}')
needs_audit_n = (~cot_df['Model_Answer'].isin(list(VALID_ANSWERS))).sum()
print(f'Missing/invalid answers: {needs_audit_n}')

for _, row in tqdm(cot_df.iterrows(), total=len(cot_df), desc='Audit'):
    key = (row['Puzzle_Number'], row['Question_Number'], row['Sample'])
    if key in audit_done:
        continue

    existing = str(row.get('Model_Answer', '') or '').strip().upper()

    # If already valid and ONLY_MISSING, keep it
    if ONLY_MISSING and existing in VALID_ANSWERS:
        audited_answer = existing
        audited_raw    = ''
        elapsed        = 0.0
    else:
        raw_output = str(row.get('Raw_Output', '') or '')
        reasoning  = str(row.get('Reasoning',  '') or '')
        user_prompt = f'Raw output:\n{raw_output}'
        if reasoning and reasoning.lower() not in ('nan', '', 'none'):
            user_prompt += f'\n\nReasoning:\n{reasoning}'

        try:
            audited_raw, elapsed = generate_audit(user_prompt)
            audited_answer = parse_answer(audited_raw)
        except Exception as e:
            print(f'  [P{row["Puzzle_Number"]} Q{row["Question_Number"]}] Audit error: {e}')
            audited_answer = 'ERROR'
            audited_raw    = f'ERROR: {e}'
            elapsed        = 0.0

    audit_records.append({
        **row.to_dict(),
        'audited_answer': audited_answer,
        'audited_raw':    audited_raw,
        'audit_elapsed':  round(elapsed, 3),
    })
    audit_done.add(key)

    if len(audit_records) % SAVE_EVERY == 0:
        pd.DataFrame(audit_records).to_csv(AUDIT_OUT, index=False)
        print(f'  [Checkpoint] {len(audit_records)} rows')

pd.DataFrame(audit_records).to_csv(AUDIT_OUT, index=False)
audit_df = pd.read_csv(AUDIT_OUT)
audit_df['correct'] = audit_df['audited_answer'] == audit_df['GT_Answer']
audit_df.to_csv(AUDIT_OUT, index=False)

print(f'\nAudit complete: {len(audit_df)} rows')
valid_n = audit_df['audited_answer'].isin(list(VALID_ANSWERS)).sum()
print(f'Valid answers  : {valid_n}/{len(audit_df)} ({valid_n/len(audit_df):.1%})')
print(f'Correct        : {audit_df["correct"].sum()}/{len(audit_df)} ({audit_df["correct"].mean():.1%})')

## 8. Load Original Results & Pair

In [ ]:
# Load existing Gemini original results — not re-run
orig_all = pd.read_csv(ORIG_AUDITED_PATH)
orig_gemini = orig_all[orig_all['model_name'] == ORIG_MODEL_NAME].copy()
orig_gemini['correct'] = orig_gemini['audited_answer'] == orig_gemini['GT_Answer']

print(f'Original Gemini rows: {len(orig_gemini)}')
print(f'Original accuracy   : {orig_gemini["correct"].mean():.1%}')
print(orig_gemini['difficulty'].value_counts().to_string())

# Load audited adapted results
audit_df = pd.read_csv(AUDIT_OUT)
audit_df['correct'] = audit_df['audited_answer'] == audit_df['GT_Answer']

print(f'\nAdapted Gemini rows : {len(audit_df)}')
print(f'Adapted accuracy    : {audit_df["correct"].mean():.1%}')
print(audit_df['difficulty'].value_counts().to_string())

# ── Pair: for each adapted puzzle, match to original solve result ─────────────
# Adapted may have multiple adaptations per (Puzzle_Number, Question_Number)
# Average adapted correctness per puzzle before pairing
adap_avg = audit_df.groupby(KEY_COLS).agg(
    correct_adap      = ('correct', 'mean'),
    difficulty        = ('difficulty', 'first'),
    Solved_Percentile = ('Solved_Percentile', 'first'),
).reset_index()

orig_slim = orig_gemini[KEY_COLS + ['correct']].rename(columns={'correct': 'correct_orig'})

paired = adap_avg.merge(orig_slim, on=KEY_COLS, how='inner')
paired['delta'] = paired['correct_adap'] - paired['correct_orig'].astype(int)

print(f'\nPaired rows         : {len(paired)}')
print(f'Orig acc (paired)   : {paired["correct_orig"].mean():.1%}')
print(f'Adap acc (paired)   : {paired["correct_adap"].mean():.1%}')
print(f'Delta               : {paired["delta"].mean()*100:+.1f} pp')

## 9. Statistical Tests

In [ ]:
from scipy.stats import ttest_rel, wilcoxon

t_stat, t_p = ttest_rel(paired['correct_adap'], paired['correct_orig'].astype(int))
try:
    w_stat, w_p = wilcoxon(paired['delta'])
except Exception:
    w_stat, w_p = float('nan'), float('nan')

sig = '***' if t_p < 0.001 else ('**' if t_p < 0.01 else ('*' if t_p < 0.05 else 'ns'))

print('='*60)
print('Paired Statistical Tests — Gemini: Original vs Adapted')
print('='*60)
print(f'n pairs          : {len(paired)}')
print(f'Orig accuracy    : {paired["correct_orig"].mean()*100:.1f}%')
print(f'Adapted accuracy : {paired["correct_adap"].mean()*100:.1f}%')
print(f'Delta            : {paired["delta"].mean()*100:+.1f} pp')
print(f'Paired t-test    : t={t_stat:.3f}, p={t_p:.4f}  {sig}')
print(f'Wilcoxon         : W={w_stat}, p={w_p:.4f}')
print()
print('By difficulty tier:')
DIFF_ORDER = ['Hard', 'Medium', 'Easy']
for diff in DIFF_ORDER:
    sub = paired[paired['difficulty'] == diff]
    if len(sub):
        print(f'  {diff:6}: orig={sub["correct_orig"].mean()*100:.1f}%  '
              f'adap={sub["correct_adap"].mean()*100:.1f}%  '
              f'Δ={sub["delta"].mean()*100:+.1f} pp  (n={len(sub)})')

# Save results
stats_out = pd.DataFrame([{
    'model': 'gemini-2.5-flash',
    'n_pairs': len(paired),
    'orig_acc_%': round(paired['correct_orig'].mean()*100, 1),
    'adap_acc_%': round(paired['correct_adap'].mean()*100, 1),
    'delta_pp': round(paired['delta'].mean()*100, 1),
    't_stat': round(t_stat, 3),
    't_p': round(t_p, 4),
    'wilcoxon_p': round(w_p, 4),
    'significant': sig,
}])
stats_out.to_csv(os.path.join(WORKDIR, 'paired_stats.csv'), index=False)
print(f'\nSaved -> paired_stats.csv')

## 10. Figures

In [ ]:
colors_diff = {'Hard': '#d62728', 'Medium': '#ff7f0e', 'Easy': '#2ca02c'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# ── Fig 1: Overall accuracy bar ───────────────────────────────────────────────
ax = axes[0]
conditions = ['Original', 'Adapted']
accs       = [paired['correct_orig'].mean()*100, paired['correct_adap'].mean()*100]
bars = ax.bar(conditions, accs, color=['#4a6fa5', '#b07d62'],
              edgecolor='#333', linewidth=0.7, width=0.4)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title(f'Gemini 2.5 Flash\nOriginal vs Adapted (n={len(paired)} pairs)\n'
             f'Paired t: t={t_stat:.2f}, p={t_p:.4f} {sig}',
             fontsize=9, fontweight='bold')
ax.set_ylim(0, 105)
ax.axhline(y=20, color='grey', linestyle='--', alpha=0.4, label='Random (20%)')
ax.legend(fontsize=8, frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle=':', linewidth=0.5, color='#ccc', zorder=0)
ax.set_axisbelow(True)

# ── Fig 2: Accuracy by difficulty tier ───────────────────────────────────────
ax = axes[1]
x     = np.arange(len(DIFF_ORDER))
width = 0.35
orig_vals = [paired[paired['difficulty']==d]['correct_orig'].mean()*100 for d in DIFF_ORDER]
adap_vals = [paired[paired['difficulty']==d]['correct_adap'].mean()*100 for d in DIFF_ORDER]

b1 = ax.bar(x - width/2, orig_vals, width, label='Original',
            color='#4a6fa5', edgecolor='#333', linewidth=0.6, alpha=0.85)
b2 = ax.bar(x + width/2, adap_vals, width, label='Adapted',
            color='#b07d62', edgecolor='#333', linewidth=0.6, alpha=0.85)

# Delta annotation
for i, diff in enumerate(DIFF_ORDER):
    sub   = paired[paired['difficulty'] == diff]
    delta = sub['delta'].mean() * 100
    ypos  = max(orig_vals[i], adap_vals[i]) + 2
    color = '#2ca02c' if delta >= 0 else '#d62728'
    ax.text(i, ypos, f'{delta:+.1f}pp', ha='center', fontsize=8,
            color=color, fontweight='bold')
    n_sub = len(sub)
    ax.text(i, -8, f'n={n_sub}', ha='center', fontsize=7.5, color='#555')

ax.set_xticks(x)
ax.set_xticklabels(DIFF_ORDER)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Accuracy by Difficulty Tier\n(Original vs Adapted)',
             fontsize=9, fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(fontsize=8, frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle=':', linewidth=0.5, color='#ccc', zorder=0)
ax.set_axisbelow(True)

plt.tight_layout(pad=1.0)
plt.savefig(os.path.join(WORKDIR, 'gemini_orig_vs_adapted.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(WORKDIR, 'gemini_orig_vs_adapted.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 11. Delta Histogram — Per-Puzzle Change

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))

# Round delta to -1, 0, +1 for discrete interpretation
# (averaged over multiple adaptations so may not be exactly -1/0/+1)
paired['delta_bin'] = paired['delta'].apply(
    lambda x: -1 if x < -0.1 else (1 if x > 0.1 else 0)
)
delta_counts = paired['delta_bin'].value_counts().sort_index()
colors_map   = {-1: '#c0392b', 0: '#95a5a6', 1: '#2980b9'}

ax.bar(delta_counts.index, delta_counts.values,
       color=[colors_map.get(i, '#888') for i in delta_counts.index],
       edgecolor='#333', linewidth=0.6, width=0.5)
ax.set_xticks([-1, 0, 1])
ax.set_xticklabels(['Orig✓ Adap✗', 'Tie', 'Orig✗ Adap✓'], fontsize=8.5)
ax.set_ylabel('Number of Puzzles', fontweight='bold')
ax.set_title('Per-Puzzle Delta: Original → Adapted\n(Gemini 2.5 Flash, CoT)', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle=':', linewidth=0.5, color='#ccc', zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'delta_histogram.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(WORKDIR, 'delta_histogram.png'), dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print('Files saved to:', WORKDIR)
for f in sorted(os.listdir(WORKDIR)):
    print(f'  {f}')